In [1]:
import glob
import json
import os
import re
from xml.etree import ElementTree as ET
# from src.data.xml_extraction import gen_xml_paths

In [2]:
!mkdir data
!mkdir data/raw
!mkdir data/external

In [5]:
field_keywords = json.load(open("data/external/field_keywords.json", encoding="utf8"))

In [6]:
def gen_xml_paths(path: str | os.PathLike) -> list[str]:
    """
    Collect xmls from a path
    Retries needed as this was originally on a network path that failed occasionally
    :param path: str : A location of transkribus model output xmls
    :return: list[str], list[str]
    """

    attempts = 0
    while attempts < 3:
        xmls = glob.glob(path)
        if xmls:
            break
        else:
            attempts += 1
            continue
    else:
        raise IOError(f"Failed to connect to {path}")

    return xmls

In [7]:
def parse_custom_attribute_string(element: ET.Element) -> list[tuple[str, tuple[str, str]]]:
    """
    Parse the custom attributes of an XML element
    Convert the custom string into a list of (Transkribus) tags and tag values

    Args:
        element (Element): _description_

    Returns:
        list[tuple[str, tuple[str, str]]]: _description_
    """
    attributes_raw = element.attrib.get("custom")
    # Handle misformatted Unicode, U+0020 (space), U+0027 (apostrophe)
    attributes = attributes_raw.replace(r"\u0020", " ").replace(r"\u0027", "'")
    attrib_pair_re = re.compile(r"(?P<tag>\w+) (?P<text>\{[\.\w\s:;\d\\'’-]+\})")
    attrib_inner_re = re.compile(r"(?P<tag>\w+):(?P<text>[\.\w\s\d\\'’-]+)")
    all_attribs = attrib_pair_re.findall(attributes)

    inner_found = [(k, attrib_inner_re.findall(v[1:-1])) for k,v in all_attribs]
    # breakpoint()
    return inner_found

In [19]:
xmls = gen_xml_paths("data/raw/0*.xml")

In [21]:
print(xmls)

['data/raw/0161_LD_31_b_`730_0267.xml', 'data/raw/0073_LD_31_b_`730_0167.xml', 'data/raw/0094_LD_31_b_`730_0188.xml', 'data/raw/0106_LD_31_b_`730_0201.xml', 'data/raw/0112_LD_31_b_`730_0211.xml', 'data/raw/0012_LD_31_b_`730_0309.xml', 'data/raw/0034_LD_31_b_`730_0128.xml', 'data/raw/0181_LD_31_b_`730_0292.xml', 'data/raw/0001_LD_31_b_`730_0298.xml', 'data/raw/0157_LD_31_b_`730_0262.xml', 'data/raw/0141_LD_31_b_`730_0240.xml', 'data/raw/0008_LD_31_b_`730_0305.xml', 'data/raw/0140_LD_31_b_`730_0239.xml', 'data/raw/0167_LD_31_b_`730_0274.xml', 'data/raw/0182_LD_31_b_`730_0293.xml', 'data/raw/0025_LD_31_b_`730_0119.xml', 'data/raw/0055_LD_31_b_`730_0149.xml', 'data/raw/0054_LD_31_b_`730_0148.xml', 'data/raw/0172_LD_31_b_`730_0283.xml', 'data/raw/0056_LD_31_b_`730_0150.xml', 'data/raw/0129_LD_31_b_`730_0228.xml', 'data/raw/0026_LD_31_b_`730_0120.xml', 'data/raw/0099_LD_31_b_`730_0193.xml', 'data/raw/0086_LD_31_b_`730_0180.xml', 'data/raw/0071_LD_31_b_`730_0165.xml', 'data/raw/0185_LD_31_b_`

In [13]:
field_keywords

{'Binding': ['Binding', 'Bound', 'Rebound', 'Inlaid'],
 'Dating': ['Dating'],
 'Provenance': ['Provenance',
  'Presented',
  'From',
  'Grenville Copy',
  'King George III’s Copy',
  'Bought']}

- iterate over all xmls
- iterate over all regions in each xml
- get the structure type for that region
- check if region is one of binding, dating, provenance
- if yes check if region starts with one of the field keywords
- if yes assign to good list
- if no assign to bad list

- iterate over regions in one xml
- get the structure type for that region
- check if region is one of binding, dating, provenance
- if yes check get keywords for field
- get all text for region
- process text to remove blank lines
- check region starts with one of the field keywords
- create good list/bad list
- assign to good/list bad list
- generalise to iterate over all xmls

In [15]:
[print(parse_custom_attribute_string(e)) for e in root[1][2:]]

[('readingOrder', [('index', '1')]), ('structure', [('score', '0.99'), ('type', 'Provenance')])]
[('readingOrder', [('index', '2')]), ('structure', [('score', '0.99'), ('type', 'table')])]
[('readingOrder', [('index', '3')]), ('structure', [('score', '0.99'), ('type', 'table')])]


[None, None, None]

In [24]:
good_regions = []
bad_regions = []
# line below is iterating over the xml
# for each iteration lets us access each text region
for xml in xmls:
    tree = ET.parse(xml)
    root = tree.getroot()
    # line below is iterating over the text regions
    for region in root[1][2:]:
        # line below getting the custom attribute string and structuring it and assigning it to attributes
        attributes = parse_custom_attribute_string(region)
        # line below is printing field type for each text region
        # print(attributes[1][1][0][1])
        # line below assigns field type to region_field_type
        region_field_type = attributes[1][1][0][1]
        # line below assigns field names for OCR that we want to add to MARC records to target_field_types
        target_field_types = ['Binding', 'Dating', 'Provenance']
        # looks for target_field_types in the region_field_type
        if region_field_type in target_field_types:
            # assigns field key words to target_field_keywords
            target_field_keywords = field_keywords.get(region_field_type)
            # creates list to gather text lines for region
            region_lines = []
            # iterating over the text lines in each text region
            for line in region[1:-1]:
                # getting the text
                line_text = line[2][0].text
                # adding the text for the line to the list of region lines
                region_lines.append(line_text)

            non_blank_lines = []
            for line in region_lines:
                if line:
                    non_blank_lines.append(line)
            for line in non_blank_lines[:4]:
                kw_present = [kw in line for kw in target_field_keywords]
                if any(kw_present):
                    good_regions.append([region_field_type, region, xml])
                    break
            else:
                bad_regions.append([region_field_type, region, xml])

print(f"Good regions: {good_regions}")
print(f"Bad regions: {bad_regions}")

Good regions: [['Provenance', <Element '{http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15}TextRegion' at 0x7bb26d381990>, 'data/raw/0012_LD_31_b_`730_0309.xml'], ['Dating', <Element '{http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15}TextRegion' at 0x7bb26d3833d0>, 'data/raw/0012_LD_31_b_`730_0309.xml'], ['Binding', <Element '{http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15}TextRegion' at 0x7bb26d3802c0>, 'data/raw/0012_LD_31_b_`730_0309.xml'], ['Provenance', <Element '{http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15}TextRegion' at 0x7bb26d40ad40>, 'data/raw/0012_LD_31_b_`730_0309.xml'], ['Binding', <Element '{http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15}TextRegion' at 0x7bb26d408590>, 'data/raw/0012_LD_31_b_`730_0309.xml'], ['Provenance', <Element '{http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15}TextRegion' at 0x7bb26d40b330>, 'data/raw/0012_LD_31_b_`730_0309.xml'], ['Provenance', <Element

In [25]:
print(f"Good regions: {len(good_regions)}")
print(f"Bad regions: {len(bad_regions)}")

Good regions: 125
Bad regions: 56


In [26]:
def print_bad_region_text(text_region: ET.Element) -> None:
    """
    Print the text of a bad region
    First extract all text lines from text region then
    print the text line by line
    :param text_region: ET.Element : A text region in a transkribus model output xmls
    :return: None
    """
    # creates list to gather text lines for region
    region_lines = []
    # iterating over the text lines in each text region
    for line in region[1:-1]:
        # getting the text
        line_text = line[2][0].text
        # adding the text for the line to the list of region lines
        region_lines.append(line_text)
    # print all lines in text region
    for line in region_lines:
        print(line)
    print("\n\n")

    return None

In [29]:
for field_type, region, xml in bad_regions:
    print(f"Bad region for {field_type}")
    print(f"Bad region for {xml}")
    print_bad_region_text(region)


Bad region for Binding
Bad region for data/raw/0012_LD_31_b_`730_0309.xml
binding signed ‘W. M.' [William Maskell].
5598; see p. 287). Affixed to the front fly-leaf is a note on the
Ecloga, printed by Pynson not later than 1498 (STC 23939.5, IA.



Bad region for Dating
Bad region for data/raw/0034_LD_31_b_`730_0128.xml
marcij respectively.
Symoni Mountfort et Emme vxori eius’ and ‘vltimo die Mensis
the date of purchase in the year 1480 have been filled in with
spaces left in I. 8 for the name of the purchaser and in l. 18 for
(1962), pp. 591-3. In this copy (the only one known) the blank
extended to 8 September 1481. See CPR 13, pp. 255, 259; Lun
term of validity of the indulgence was at first Easter 1481, later
was granted by Sixtus IV in a bull dated 12 December 1479. The
Dating This indulgence in favour of the Knights of St John



Bad region for Dating
Bad region for data/raw/0001_LD_31_b_`730_0298.xml



Bad region for Binding
Bad region for data/raw/0025_LD_31_b_`730_0119.xml
ba

In [24]:
name = "Jeanette"
print(f"hello {name}")

hello Jeanette
